# Demo — Running Medallion Lakehouse Governance as an Ongoing Operation
`lh.medallion_governance_audit()` combines signals from every layer of Veridian Health's Medallion Lakehouse — Glue catalog gaps (`MP2`), Silver contract validation (`MP3`), six-dimension quality checks (`MP4`), OpenLineage coverage (`MP6`), and Lake Formation PHI access enforcement (`MP7`) — into a single operational findings table.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your AWS Cloud Lab environment and seed the **Veridian Health Medallion Lakehouse** (`veridian_lakehouse` in AWS Glue, S3 `bronze/silver/gold` Parquet prefixes, and AWS Lake Formation enforcement) via `setup_aws.py`.

In [1]:
import os
os.environ["VERIDIAN_LF_LIVE"] = "1"
os.environ["VERIDIAN_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
glue = boto3.client("glue", region_name=os.environ["VERIDIAN_AWS_REGION"])
try:
    glue.get_database(Name="veridian_lakehouse")
    print("Glue database 'veridian_lakehouse' is already seeded and ready.")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

Glue database 'veridian_lakehouse' is already seeded and ready.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from demo_toolkit import connect_lakehouse, lh_backend

lh = connect_lakehouse()
findings = lh.medallion_governance_audit(contracts_valid=True)
print(f"Backend: {lh_backend()} -> {type(lh).__name__} | Total findings: {len(findings)}")
display(findings.groupby(["signal", "severity"]).size().unstack(fill_value=0))
display(findings)

Backend: live -> AwsMedallionLakehouse | Total findings: 23

severity                high  low  medium
signal                                   
authorization_conflict     1    0       0
missing_lineage            0    0       2
missing_metadata           3    0       6
phi_access_gap             2    0       0
quality_violation          5    4       0


                    signal    zone                                                                                          object severity                         source
0         missing_metadata  silver                                                           silver_patient_master.mrn (type TEXT)     high  audit_lakehouse_catalog (MP2)
1         missing_metadata  bronze                                                          bronze_patient_intake.retention_policy   medium  audit_lakehouse_catalog (MP2)
2         missing_metadata  silver                                                               silver_clinical_encounter.steward   medium  audit_lakehouse_catalog (MP2)
3         missing_metadata  silver                                                                   silver_lab_result.sensitivity   medium  audit_lakehouse_catalog (MP2)
4         missing_metadata  bronze                                                  bronze_lab_device_calibration.retention_policy   medium  audi

## 1. Close one Glue catalog gap and one Lake Formation PHI access gap

In [3]:
# 1) Assign the Clinical Telehealth steward on silver_clinical_encounter in AWS Glue
lh.set_table_governance("silver_clinical_encounter", steward="clinical-ops@veridian.health")

# 2) Attach LF-Tag phi_governed=true to silver_patient_master and grant vh_him_steward access by tag
lh.tag_medallion_table("silver_patient_master", "phi_governed", "true")
lh.grant_lf_tag("vh_him_steward", "phi_governed", "true")

after = lh.medallion_governance_audit(contracts_valid=True)
print(f"Findings before: {len(findings)} -> after fixing 1 Glue gap + 1 LF PHI table: {len(after)}")
display(after.groupby(["signal", "severity"]).size().unstack(fill_value=0))

Findings before: 23 -> after fixing 1 Glue gap + 1 LF PHI table: 21


severity                high  low  medium
signal                                   
authorization_conflict     1    0       0
missing_lineage            0    0       2
missing_metadata           3    0       5
phi_access_gap             1    0       0
quality_violation          5    4       0
